# Cross-Platform DRL Optimizer (XPCO) Demo

Demonstrates **multi-platform** portfolio optimization through the full pipeline:
1. Build the optimizer stack: SAC → SafeDRL → HybridDRLLLM → CrossPlatformOptimizer
2. Seed historical data so the marginal ROAS estimator can differentiate platforms
3. Create campaign states and contexts for 5 platforms with realistic spend/revenue
4. Run `CrossPlatformOptimizer.optimize_portfolio()` — budget allocation + per-campaign DRL + mock LLM + xAI
5. Inspect: budget allocations, per-campaign actions, mock creative, xAI narratives
6. Audience segment allocation

**Key difference from SPCO:** Cross-platform estimates marginal ROAS per platform, solves budget allocation, THEN runs per-campaign DRL on each platform.

In [1]:
# ── Cell 1: Setup & Imports ───────────────────────────────────
# Inline comments explain each import's purpose and where it is used in this notebook.

import sys          # sys.path: add parent dir so 'drl' package is importable
import os           # os.path.join: build trained model path
import asyncio      # optimize_portfolio() and optimize() are async — Jupyter auto-await
import random       # (optional) for reproducibility if needed
import numpy as np  # np.random: seed history in Cell 3; np used in state_action internally
import pandas as pd # pd.DataFrame: display allocation table (Cell 6), per-campaign actions (Cell 7)
import torch        # PyTorch: SAC neural networks loaded via sac_agent

# Parent directory must contain the 'drl' package (sac_agent, hybrid_optimizer, etc.)
DRL_ROOT = '/Users/avikakhemuka/Downloads'
sys.path.insert(0, DRL_ROOT)

# load_sac_for_inference: just the function/tool that knows HOW to load a model.
from drl.sac_agent import load_sac_for_inference

# SafeDRLAgent: wraps SAC with guardrails (±50% bid, ±30% budget) — used in Cell 2
# CampaignContext: guardrail metadata per campaign — used in Cell 4 when building campaigns
from drl.safe_agent import SafeDRLAgent, CampaignContext

# HybridDRLLLMOptimizer: full pipeline (DRL + mock LLM + xAI) — used in Cell 2
# LLMClient: mock LLM returning placeholder ad copy — passed to HybridDRLLLMOptimizer
from drl.hybrid_optimizer import HybridDRLLLMOptimizer, LLMClient

# CrossPlatformOptimizer: portfolio budget allocation + per-campaign DRL — used in Cells 2, 5
from drl.cross_platform_optimizer import CrossPlatformOptimizer

# CampaignState: 39-dim state vector — used in Cell 4 when building each platform's state
# ActionSpace, AudienceAction, CreativeAction: action types — used in Cells 6, 7, 10
from drl.state_action import CampaignState, ActionSpace, AudienceAction, CreativeAction

# AudienceConstraintManager, AudienceSegment: segment budget allocation — used in Cell 10
from drl.audience_constraints import AudienceConstraintManager, AudienceSegment

'''AudienceConstraintManager
This manages second-level budget allocation - splitting a platform's budget across audience segments.'''


# Path to trained SAC agent model (contains agent.pt with learned weights)
# Path to 39-dim SAC checkpoint. Run 'python -m drl.train' to create checkpoints/final_model
TRAINED_MODEL_DIR = os.path.join(DRL_ROOT, 'drl', 'checkpoints', 'final_model')

print('Imports OK')

ModuleNotFoundError: No module named 'torch'

In [ ]:
# ── Cell 2: Build the Full Optimizer Stack ────────────────────
# Stack: SACAgent → SafeDRLAgent → HybridDRLLLMOptimizer → CrossPlatformOptimizer

# Step 1: Load trained SAC agent from model directory (39-dim state, 2 continuous + 2 discrete action heads)
agent, ckpt_features = load_sac_for_inference(
    model_dir=TRAINED_MODEL_DIR,    # directory containing agent.pt with learned weights
    state_dim=39,                   # 33 base + 3 spend + 3 audience features
    continuous_action_dim=2,        # bid_adjustment, budget_adjustment
    discrete_action_dims=[4, 4],    # 4 audience actions (HOLD/EXPAND/REFINE/EXCLUDE), 4 creative
    hidden_dim=256,                 # hidden layer size in Actor/Critic networks
    device='cpu',                   # run on CPU (use 'cuda' for GPU)
)
agent.actor.eval()  # eval mode: disables dropout for deterministic inference

# Step 2: Wrap with SafeDRLAgent (clips bid ±50%, budget ±30%, enforces cooldowns)
safe_agent = SafeDRLAgent(agent)

# Step 3: HybridDRLLLMOptimizer with mock LLM (produces DRL action + directive + mock creative + xAI)
hybrid_optimizer = HybridDRLLLMOptimizer(
    drl_agent=safe_agent,           # SafeDRLAgent wrapping the SAC agent
    llm_client=LLMClient(),         # mock LLM — _generate_mock_response() returns placeholder ad copy
    enable_tactical=True,           # enable LLM creative generation (headline, body, CTA)
)

# Step 4: CrossPlatformOptimizer — adds portfolio-level budget allocation on top
xp_optimizer = CrossPlatformOptimizer(
    hybrid_optimizer=hybrid_optimizer,  # calls optimize() for each campaign
    max_concurrent_campaigns=10,       # max parallel campaigns when running per-campaign DRL
)

print(f'Cross-platform optimizer stack built:')
print(f'  SACAgent:              state_dim={agent.config.state_dim}')
print(f'  SafeDRLAgent:          guardrails active')
print(f'  HybridDRLLLMOptimizer: mock LLM + xAI enabled')
print(f'  CrossPlatformOptimizer: budget allocation + per-campaign DRL')

Cross-platform optimizer stack built:
  SACAgent:              state_dim=39
  SafeDRLAgent:          guardrails active
  HybridDRLLLMOptimizer: mock LLM + xAI enabled
  CrossPlatformOptimizer: budget allocation + per-campaign DRL


### Reading the output above

Four layers stacked:
1. **SACAgent** (39-dim) — neural network producing raw actions.
2. **SafeDRLAgent** — clips actions to safe bounds (±50% bid, ±30% budget).
3. **HybridDRLLLMOptimizer** — DRL action + directive + mock LLM creative + xAI narrative.
4. **CrossPlatformOptimizer** — estimates marginal ROAS per platform → solves convex budget allocation → runs per-campaign DRL with adjusted budgets.

The cross-platform optimizer calls `HybridDRLLLMOptimizer.optimize()` for EACH campaign, so every campaign gets its own DRL action, mock LLM creative, and xAI narrative.

In [ ]:
# ── Cell 3: Seed Historical Data ──────────────────────────────
# The MarginalReturnEstimator needs historical (spend, revenue) observations
# per platform to differentiate marginal ROAS. Without this, it returns
# identical estimates for all platforms → flat 20/20/20/20/20 allocation.
#
# In production, this history comes from daily pipeline runs.
# For demo, we simulate 30 days of platform-specific performance.

np.random.seed(42)  # reproducibility for demo

# Platform characteristics for history simulation:
#   (avg_daily_spend, avg_roas, spend_volatility)
history_profiles = {
    'google':   (5000,  6.0, 0.10),   # stable, high ROAS
    'meta':     (8000,  3.5, 0.20),   # high spend, mediocre ROAS, volatile
    'tiktok':   (3000,  3.0, 0.25),   # small budget, inconsistent
    'amazon':   (6000,  8.0, 0.08),   # best ROAS, very stable
    'linkedin': (3000,  2.2, 0.15),   # expensive B2B, low ROAS
}

for day in range(30):  # 30 days of synthetic history per platform
    for platform, (avg_spend, avg_roas, vol) in history_profiles.items():
        daily_spend = avg_spend * (1 + np.random.normal(0, vol))  # add noise (vol = volatility)
        daily_revenue = daily_spend * avg_roas * (1 + np.random.normal(0, vol * 0.5))  # revenue with smaller noise
        daily_spend = max(daily_spend, 100)    # floor: avoid zero/negative spend
        daily_revenue = max(daily_revenue, 0)  # floor: avoid negative revenue

        xp_optimizer.estimator.record_observation(  # one observation per platform per day
            platform=platform,
            daily_spend=daily_spend,
            daily_revenue=daily_revenue,
        )

# Verify the estimator now has differentiated estimates (Amazon highest, LinkedIn lowest) Location:cross_platform_optimizer.py lines 381-395
print('Marginal ROAS estimates (after seeding 30 days of history):')
for platform in history_profiles:
    mroas, conf = xp_optimizer.estimator.estimate(platform, history_profiles[platform][0])
    print(f'  {platform:10s}: marginal_roas={mroas:.2f}x  confidence={conf:.2f}')

Marginal ROAS estimates (after seeding 30 days of history):
  google    : marginal_roas=5.30x  confidence=0.95
  meta      : marginal_roas=3.48x  confidence=0.91
  tiktok    : marginal_roas=2.69x  confidence=0.89
  amazon    : marginal_roas=8.01x  confidence=0.96
  linkedin  : marginal_roas=2.06x  confidence=0.90


### Reading the output above

The estimator now has 30 days of (spend, revenue) per platform. The marginal ROAS estimates should reflect the platform characteristics:
- **Amazon** — highest marginal ROAS (~7-8x). The allocator should shift budget here.
- **Google** — strong (~5-6x). Should maintain or gain budget.
- **Meta/TikTok** — moderate (~3x). Budget may decrease.
- **LinkedIn** — lowest (~2x). Should lose budget.

Confidence is higher when the platform has more data and less volatility in its ROAS.

In [ ]:
# ── Cell 4: Build 5-Platform Campaign Data ────────────────────
# Each campaign_info dict MUST include spend, revenue, conversions, clicks,
# impressions — these feed into PlatformMetrics for budget allocation.
# Without them, all platforms show $0 spend → flat 20% allocation.

# platform_configs: list of dicts — each platform's metrics (normalized + raw) for CampaignState + allocation
platform_configs = [
    # Google: normalized metrics (0–1) for SAC; raw for guardrails; spend/revenue for allocator
    {
        'name': 'google', 'platform_enc': 0.0,  # platform_enc: 0=google, 0.25=meta, 0.5=tiktok, 0.75=amazon, 1=linkedin
        'ctr': 0.035, 'cvr': 0.04, 'roas': 0.60, 'cpa': 0.55,  # normalized (0–1) — fed to SAC
        'cpc': 0.20, 'cpm': 0.25, 'fatigue': 0.40, 'roas_trend': 0.08,
        'raw_roas': 6.0, 'raw_cpa': 28.0, 'budget': 5000, 'bid': 2.50,  # raw — for guardrails & display
        'product': 'Enterprise SaaS Platform',
        'spend': 4200, 'revenue': 25200, 'conversions': 150,  # required for budget allocation
        'clicks': 3500, 'impressions': 100000,
    },
    # Meta: high fatigue (0.78), declining ROAS — allocator should reduce budget
    {
        'name': 'meta', 'platform_enc': 0.25,
        'ctr': 0.055, 'cvr': 0.03, 'roas': 0.45, 'cpa': 0.50,
        'cpc': 0.12, 'cpm': 0.15, 'fatigue': 0.78, 'roas_trend': -0.05,
        'raw_roas': 3.5, 'raw_cpa': 53.0, 'budget': 8000, 'bid': 1.80,
        'product': 'DTC Skincare Line',
        'spend': 7200, 'revenue': 25200, 'conversions': 136,
        'clicks': 8000, 'impressions': 145000,
    },
    # TikTok: high CTR, low CVR — audience refinement likely
    {
        'name': 'tiktok', 'platform_enc': 0.5,
        'ctr': 0.070, 'cvr': 0.015, 'roas': 0.35, 'cpa': 0.40,
        'cpc': 0.08, 'cpm': 0.10, 'fatigue': 0.55, 'roas_trend': 0.15,
        'raw_roas': 3.0, 'raw_cpa': 50.0, 'budget': 3000, 'bid': 1.20,
        'product': 'Trendy Fitness App',
        'spend': 2600, 'revenue': 7800, 'conversions': 52,
        'clicks': 4200, 'impressions': 60000,
    },
    # Amazon: best ROAS (8x) — allocator should increase budget
    {
        'name': 'amazon', 'platform_enc': 0.75,
        'ctr': 0.040, 'cvr': 0.06, 'roas': 0.80, 'cpa': 0.65,
        'cpc': 0.15, 'cpm': 0.18, 'fatigue': 0.30, 'roas_trend': 0.10,
        'raw_roas': 8.0, 'raw_cpa': 20.0, 'budget': 6000, 'bid': 2.00,
        'product': 'Organic Protein Powder',
        'spend': 5400, 'revenue': 43200, 'conversions': 270,
        'clicks': 5000, 'impressions': 125000,
    },
    # LinkedIn: lowest ROAS (2.2x), highest CPA — allocator should decrease budget
    {
        'name': 'linkedin', 'platform_enc': 1.0,
        'ctr': 0.018, 'cvr': 0.025, 'roas': 0.30, 'cpa': 0.30,
        'cpc': 0.35, 'cpm': 0.40, 'fatigue': 0.45, 'roas_trend': -0.02,
        'raw_roas': 2.2, 'raw_cpa': 95.0, 'budget': 3000, 'bid': 5.00,
        'product': 'B2B Consulting Services',
        'spend': 2800, 'revenue': 6160, 'conversions': 29,
        'clicks': 900, 'impressions': 50000,
    },
]

# Optional: override platform metrics for quick what-if tests. Keys: platform index (0=google, 1=meta, 2=tiktok, 3=amazon, 4=linkedin)
# Example: override_edits = {3: {"roas": 0.2, "raw_roas": 0.5, "raw_cpa": 95}}  # make Amazon struggling
override_edits = {}
'''override_edits = {
    0: {  # Google is index 0
        "roas": 0.35,        # was 0.60 — lower normalized ROAS
        "raw_roas": 3.0,     # was 6.0 — worse ROAS
        "raw_cpa": 45.0,     # was 28 — higher CPA
        "spend": 5000,       # same or higher spend
        "revenue": 15000,    # LOWER revenue (so ROAS = 15000/5000 = 3x instead of 6x)
    }
}'''

if override_edits:
    for row_id, edits in override_edits.items():
        for col, val in edits.items():
            platform_configs[int(row_id)][col] = val
    print(f"Applied override_edits: {override_edits}")

# Build (state, context, info) tuples — format required by optimize_portfolio()
campaigns = []
for i, p in enumerate(platform_configs):  # one campaign per platform for demo
    state = CampaignState(  # 39-dim normalized vector — input to SAC agent
        platform=p['name'], campaign_id=f'camp_{p["name"]}_{i}',
        ctr=p['ctr'], cvr=p['cvr'], roas=p['roas'], cpa=p['cpa'],
        cpc=p['cpc'], cpm=p['cpm'],
        spend_velocity=0.70, impression_volume=0.25,
        click_volume=0.12, conversion_volume=0.06,
        hour_of_day=0.5, day_of_week=0.3, day_of_month=0.5,
        is_weekend=0.0, is_holiday=0.0, days_remaining=0.33,
        ctr_trend_7d=0.02, cvr_trend_7d=0.01,
        roas_trend_7d=p['roas_trend'], cpa_trend_7d=-0.03,
        spend_trend_7d=0.05,
        impression_share=0.40, auction_pressure=0.50, competitive_position=0.45,
        audience_quality_score=0.60, creative_fatigue_score=p['fatigue'],
        predicted_cvr=p['cvr'] + 0.005, predicted_ltv=0.50,
        propensity_score=0.50,
        optimization_goal_encoding=0.0,
        platform_encoding=p['platform_enc'],
        campaign_maturity=0.15, budget_utilization=0.75,
        log_daily_spend=0.40, log_total_campaign_spend=0.25,
        log_daily_budget=0.45,
        segment_count=2, top_segment_roas=p['roas'], avg_frequency=2.0,
    )
    context = CampaignContext(  # guardrail metadata — SafeDRLAgent uses for clipping (not fed to SAC)
        campaign_id=f'camp_{p["name"]}_{i}',
        current_bid=p['bid'], current_budget=float(p['budget']),
        last_action_at=None, actions_today=0,
        current_roas=p['raw_roas'], current_cpa=p['raw_cpa'],
        target_cpa=p['raw_cpa'] * 1.2, min_roas=2.0,
        total_spend=float(p['spend']),
    )
    info = {  # campaign_info: MUST include spend/revenue/conversions — allocator uses for budget shifts
        'platform': p['name'],
        'product_name': p['product'],
        'target_audience': 'general audience',
        'spend': p['spend'],
        'revenue': p['revenue'],
        'conversions': p['conversions'],
        'clicks': p['clicks'],
        'impressions': p['impressions'],
    }
    campaigns.append((state, context, info))

TOTAL_BUDGET = sum(p['budget'] for p in platform_configs)  # $25k total portfolio

print(f'Created {len(campaigns)} campaigns across 5 platforms')
print(f'Total portfolio budget: ${TOTAL_BUDGET:,.0f}')
print(f'{"":-<72}')
print(f'{"Platform":10s} {"Budget":>8s} {"Spend":>8s} {"Revenue":>10s} {"ROAS":>6s} {"CPA":>6s} {"Fatigue":>8s}')
print(f'{"":-<72}')
for p in platform_configs:
    roas = p['revenue'] / max(p['spend'], 1)
    print(f'{p["name"]:10s} ${p["budget"]:>6,} ${p["spend"]:>6,} ${p["revenue"]:>8,} {roas:>5.1f}x ${p["raw_cpa"]:>5.0f} {p["fatigue"]:>7.2f}')

Created 5 campaigns across 5 platforms
Total portfolio budget: $25,000
------------------------------------------------------------------------
Platform     Budget    Spend    Revenue   ROAS    CPA  Fatigue
------------------------------------------------------------------------
google     $ 5,000 $ 4,200 $  25,200   6.0x $   28    0.40
meta       $ 8,000 $ 7,200 $  25,200   3.5x $   53    0.78
tiktok     $ 3,000 $ 2,600 $   7,800   3.0x $   50    0.55
amazon     $ 6,000 $ 5,400 $  43,200   8.0x $   20    0.30
linkedin   $ 3,000 $ 2,800 $   6,160   2.2x $   95    0.45


### Reading the output above

Each campaign now has **real spend and revenue numbers** (not just normalised state features). This is critical — the `PlatformPerformanceTracker` uses these to compute budget shares and the allocator uses them to decide shifts.

**Quick scenario test:** Set `override_edits = {3: {"roas": 0.2, "raw_roas": 0.5}}` to make Amazon struggling, then re-run Cell 4 and Cell 5.

Platform characteristics:
- **Amazon** — $5.4k spend generating $43k revenue (8.0x ROAS). Best performer → should gain budget.
- **Google** — $4.2k spend, $25k revenue (6.0x). Strong → maintain or gain.
- **Meta** — $7.2k spend, $25k revenue (3.5x). Biggest spender but mediocre ROAS + high fatigue → should lose budget.
- **TikTok** — $2.6k spend, $7.8k revenue (3.0x). Small and inconsistent.
- **LinkedIn** — $2.8k spend, $6.2k revenue (2.2x). Most expensive per conversion → should lose budget.

In [ ]:
# ── Cell 5: Run Cross-Platform Optimization ───────────────────
# optimize_portfolio() runs the FULL pipeline:
#   1. Build portfolio snapshot (aggregate spend/revenue by platform)
#   2. Estimate marginal ROAS per platform (using seeded history)
#   3. Solve budget allocation (convex optimization — shift to high-ROAS platforms)
#   4. For EACH campaign: run HybridDRLLLMOptimizer.optimize()
#      → SAC action + guardrails + directive + mock LLM creative + xAI narrative
#   5. Return CrossPlatformResult with allocations + per-campaign results

# optimize_portfolio: async — Jupyter auto-await; returns CrossPlatformResult
result = await xp_optimizer.optimize_portfolio(
    organization_id='org_demo_001',  # for tracker/cooldown key (prevents too-frequent rebalances)
    campaigns=campaigns,             # list of (state, context, info) from Cell 4
    total_budget=TOTAL_BUDGET,       # $25k total portfolio
    force_rebalance=True,            # skip cooldown check for demo (always rebalance)
)

print(f'=== PORTFOLIO SUMMARY ===')
print(f'  Portfolio ROAS:          {result.portfolio_roas:.2f}x')   # weighted avg across platforms
print(f'  Total budget:            ${result.total_budget:,.0f}')
print(f'  Total spend:             ${result.total_spend:,.0f}')
print(f'  Rebalance triggered:     {result.rebalance_triggered}')   # True = optimizer shifted budget
print(f'  Allocation confidence:   {result.allocation_confidence:.2%}')  # marginal ROAS confidence × diversification
print(f'  Projected portfolio ROAS:{result.projected_portfolio_roas:.2f}x')  # after applying recommended shifts
if result.blocked_reason:
    print(f'  Note: {result.blocked_reason}')

=== PORTFOLIO SUMMARY ===
  Portfolio ROAS:          4.85x
  Total budget:            $25,000
  Total spend:             $22,200
  Rebalance triggered:     True
  Allocation confidence:   92.54%
  Projected portfolio ROAS:4.97x


### Reading the output above

- **Portfolio ROAS** — weighted average ROAS across all platforms (based on actual spend/revenue).
- **Rebalance triggered = True** — the optimizer is actively shifting budget between platforms.
- **Projected portfolio ROAS** — estimated ROAS after applying the recommended budget shifts. Should be higher than current.
- **Allocation confidence** — average of per-platform marginal ROAS confidence × diversification score.

In [ ]:
# ── Cell 6: Budget Allocation Recommendations ─────────────────
# result.allocations contains one AllocationRecommendation per platform.
# This shows HOW the optimizer wants to redistribute the portfolio budget
# (shift % from low-ROAS platforms to high-ROAS platforms).

if result.allocations:
    alloc_data = []
    for a in result.allocations:  # one AllocationRecommendation per platform
        alloc_data.append({
            'Platform': a.platform,
            'Current Share': f'{a.current_share:.1%}',
            'Recommended': f'{a.recommended_share:.1%}',
            'Shift': f'{a.shift_pct:+.1%}',  # positive = increase, negative = decrease
            'Current $': f'${a.current_budget:,.0f}',
            'Recommended $': f'${a.recommended_budget:,.0f}',
            'Change $': f'${a.recommended_budget - a.current_budget:+,.0f}',
            'Confidence': f'{a.confidence:.0%}',
        })
    print('\BUDGET ALLOCATION RECOMMENDATIONS ')
    display(pd.DataFrame(alloc_data))

    print('\nRationales:')
    for a in result.allocations:
        direction = 'INCREASE' if a.shift_pct > 0.005 else ('DECREASE' if a.shift_pct < -0.005 else 'MAINTAIN')
        print(f'  {a.platform:10s} [{direction}] {a.rationale}')
else:
    print('No allocations (single-platform passthrough or allocation skipped)')

\BUDGET ALLOCATION RECOMMENDATIONS 


,Platform,Current Share,Recommended,Shift,Current $,Recommended $,Change $,Confidence
0,google,18.9%,22.9%,+4.0%,"$4,730","$5,733","$+1,003",96%
1,meta,32.4%,15.8%,-16.7%,"$8,108","$3,943","$-4,165",91%
2,tiktok,11.7%,12.9%,+1.2%,"$2,928","$3,232",$+304,89%
3,amazon,24.3%,37.1%,+12.8%,"$6,081","$9,279","$+3,198",97%
4,linkedin,12.6%,11.3%,-1.4%,"$3,153","$2,813",$-340,90%



Rationales:
  google     [INCREASE] Increase google allocation by 4.0%. because: strong marginal ROAS (5.50x), 150 conversions
  meta       [DECREASE] Decrease meta allocation by 16.7%. to reallocate to higher-performing platforms
  tiktok     [INCREASE] Increase tiktok allocation by 1.2%. because: strong marginal ROAS (2.78x), 52 conversions
  amazon     [INCREASE] Increase amazon allocation by 12.8%. because: strong marginal ROAS (7.99x), 270 conversions
  linkedin   [DECREASE] Decrease linkedin allocation by 1.4%. to reallocate to higher-performing platforms


### Reading the allocation table above

**What to look for:**
- **Amazon** (ROAS 8x) should have the largest positive shift — the allocator wants to pour more money into the best-performing platform.
- **Google** (ROAS 6x) should also gain budget.
- **Meta** (ROAS 3.5x, high fatigue) should lose budget — it's the biggest spender but mediocre returns.
- **LinkedIn** (ROAS 2.2x) should lose the most — worst ROAS.
- **Shift** is capped at `max_platform_shift_pct` (default 15%) to prevent drastic changes.

The "Change $" column shows the actual dollar impact of the recommended shift.

In [ ]:
# ── Cell 7: Per-Campaign DRL Actions + Mock LLM Creative ──────
# result.platform_campaign_results: { platform: [OptimizationResult, ...] }
# Each OptimizationResult = DRL action + directive + mock LLM creative + xAI narrative

action_rows = []

# platform_campaign_results: one OptimizationResult per campaign (from HybridDRLLLMOptimizer.optimize)
for platform, opt_results in result.platform_campaign_results.items():
    for opt in opt_results:  # OptimizationResult from HybridDRLLLMOptimizer.optimize()
        a = opt.action   # SAC output (bid/budget adj, audience/creative action)
        d = opt.directive  # DRLDirective (tone, max_offer_discount)
        t = opt.tactical   # TacticalExecution (headline, body, CTA from mock LLM)

        action_rows.append({
            'Platform': platform,
            'Bid Adj': f'{a.bid_adjustment:+.2%}',      # continuous action, clipped by SafeDRLAgent
            'Budget Adj': f'{a.budget_adjustment:+.2%}',
            'Audience': AudienceAction(a.audience_action).name,   # HOLD/EXPAND/REFINE/EXCLUDE
            'Creative': CreativeAction(a.creative_action).name,   # HOLD/ROTATE/PAUSE/TEST
            'Confidence': f'{opt.combined_confidence:.1%}',       # 0.7*strategic + 0.3*tactical
            'Tone': d.messaging_tone,                  # aggressive_growth, efficiency_focused, etc.
            'Headline': t.headline if t else 'N/A',     # mock LLM output
            'CTA': t.call_to_action if t else 'N/A',
            'Max Discount': f'{d.max_offer_discount:.0%}',  # derived from ROAS (5%/15%/25%)
            'Review?': 'YES' if opt.requires_review else 'no',  # human approval if confidence < 60%
        })

print(' PER-CAMPAIGN ACTIONS + MOCK LLM CREATIVE ')
display(pd.DataFrame(action_rows))

 PER-CAMPAIGN ACTIONS + MOCK LLM CREATIVE 


,Platform,Bid Adj,Budget Adj,Audience,Creative,Confidence,Tone,Headline,CTA,Max Discount,Review?
0,google,+18.24%,+20.71%,REFINE,HOLD,64.1%,urgency,Don't Miss Out - Limited Time Offer!,Shop Now,25%,YES
1,meta,-22.72%,-27.20%,REFINE,PAUSE_UNDERPERFORMING,64.1%,efficiency_focused,Don't Miss Out - Limited Time Offer!,Shop Now,25%,YES
2,tiktok,-12.57%,-5.65%,REFINE,PAUSE_UNDERPERFORMING,64.0%,consistent,Don't Miss Out - Limited Time Offer!,Shop Now,25%,YES
3,amazon,-4.98%,+5.38%,REFINE,HOLD,64.1%,consistent,Don't Miss Out - Limited Time Offer!,Shop Now,25%,YES
4,linkedin,-7.93%,+4.57%,REFINE,PAUSE_UNDERPERFORMING,64.0%,consistent,Don't Miss Out - Limited Time Offer!,Shop Now,15%,YES


### Reading the per-campaign table above

Each row is one campaign. Every column is an output of the **full `HybridDRLLLMOptimizer.optimize()` pipeline**:

| Column | Source | Meaning |
|--------|--------|---------|
| Bid/Budget Adj | SAC agent → SafeDRLAgent | Continuous action, clipped by guardrails |
| Audience/Creative | SAC agent | Discrete action (HOLD/EXPAND/REFINE/EXCLUDE, HOLD/ROTATE/PAUSE/TEST) |
| Confidence | 0.7\*strategic + 0.3\*tactical | Below 60% → requires human review |
| Tone | `_derive_directive()` | Messaging tone for LLM (aggressive_growth, efficiency_focused, etc.) |
| Headline / CTA | `_generate_mock_response()` | Mock LLM ad copy (placeholder in demo) |
| Max Discount | Derived from ROAS | 5% if ROAS < 1.5, 15% if < 3.0, 25% if >= 3.0 |
| Review? | Confidence + guardrails | YES = human must approve before deploying |

In [ ]:
# ── Cell 8: xAI Narratives Per Platform ───────────────────────
# These are generated INSIDE HybridDRLLLMOptimizer.optimize() by OptimizationNarrator.
# Each narrative explains: situation, decision, reasoning, confidence, reasonability.

for platform, opt_results in result.platform_campaign_results.items():
    for opt in opt_results:  # OptimizationResult from HybridDRLLLMOptimizer.optimize()
        n = opt.narrative
        if n:
            print(f'\n{"=" * 60}')
            print(f'xAI NARRATIVE: {platform.upper()}')
            print(f'{"=" * 60}')
            print(f'SITUATION: {n["situation"]}')
            print(f'DECISION:  {n["decision"]}')
            print('REASONING:')
            for bullet in n['reasoning']:
                print(f'  * {bullet}')
            print(f'CONFIDENCE: {n["confidence"]}')
            print(f'REASONABILITY: {n["reasonability"]}')


xAI NARRATIVE: GOOGLE
SITUATION: The campaign is showing no strongly notable signals.
DECISION:  The model recommends increasing the bid by 18%, scaling the daily budget up by 21% and refineing the audience.
REASONING:
  * • Impression share 40% → increase bid +18% because capturing more impression share accelerates reach.
  * • ROAS trend +8% → scale budget +21% because positive momentum signals room to scale spend.
  * • CVR or auction pressure elevated → audience: REFINE because tightening targeting improves conversion efficiency.
CONFIDENCE: The model is only marginally confident at 49% because the state signals are mixed and this is a young campaign (maturity 0.15).
REASONABILITY: A 18% bid change and a 21% budget change are both within safe guardrail limits (max ±50% bid, max ±30% budget). Human review is recommended due to moderate confidence.

xAI NARRATIVE: META
SITUATION: The campaign is showing elevated creative fatigue at 0.78.
DECISION:  The model recommends decreasing th

### Reading the xAI narratives above

Each platform gets its own narrative because `optimize_portfolio()` calls `HybridDRLLLMOptimizer.optimize()` separately per campaign. The narrative connects platform-specific state signals to actions:

- **Meta** — should mention elevated creative fatigue (0.78) and declining ROAS trend → creative rotation.
- **Amazon** — should note strong ROAS + positive trend → scale budget.
- **LinkedIn** — should flag high CPA → efficiency-focused tone.
- **TikTok** — should note high CTR but low CVR → audience refinement.

In [ ]:
# ── Cell 9: Mock LLM Creative Detail ──────────────────────────
# Show the full TacticalExecution output for each platform.
# In demo: LLMClient returns placeholder copy. In prod: real LLM generates platform-specific ads.

for platform, opt_results in result.platform_campaign_results.items():
    for opt in opt_results:  # OptimizationResult from HybridDRLLLMOptimizer.optimize()
        t = opt.tactical
        d = opt.directive
        if t:
            print(f'\n── {platform.upper()} Mock LLM Creative ──')
            print(f'  Directive tone:    {d.messaging_tone}')
            print(f'  Headline:          {t.headline}')
            print(f'  Body copy:         {t.body_copy}')
            print(f'  CTA:               {t.call_to_action}')
            print(f'  Offer:             {t.offer_text}')
            print(f'  Highlights:        {t.product_highlights}')
            print(f'  Urgency elements:  {t.urgency_elements}')
            print(f'  Model used:        {t.model_used}')
        else:
            print(f'\n── {platform.upper()}: No tactical output ──')


── GOOGLE Mock LLM Creative ──
  Directive tone:    urgency
  Headline:          Don't Miss Out - Limited Time Offer!
  Body copy:         Discover premium quality at unbeatable prices. Shop now and save big on your favorite items.
  CTA:               Shop Now
  Offer:             Save 15% Today
  Highlights:        ['Premium Quality', 'Fast Shipping', 'Easy Returns']
  Urgency elements:  ['Limited Time', 'While Supplies Last']
  Model used:        gpt-4

── META Mock LLM Creative ──
  Directive tone:    efficiency_focused
  Headline:          Don't Miss Out - Limited Time Offer!
  Body copy:         Discover premium quality at unbeatable prices. Shop now and save big on your favorite items.
  CTA:               Shop Now
  Offer:             Save 15% Today
  Highlights:        ['Premium Quality', 'Fast Shipping', 'Easy Returns']
  Urgency elements:  ['Limited Time', 'While Supplies Last']
  Model used:        gpt-4

── TIKTOK Mock LLM Creative ──
  Directive tone:    consistent
  Hea

### Reading the mock LLM creative above

Each platform gets a `TacticalExecution` generated by `LLMClient._generate_mock_response()`. In the demo, the mock always returns the same placeholder creative. In production with a real LLM API key:
- The **directive tone** (aggressive_growth, efficiency_focused, etc.) would be included in the prompt.
- The LLM would generate platform-specific, audience-tailored ad copy.
- The **headline** would be under 60 characters.
- The **offer** would respect the `max_offer_discount` cap from the directive.

In [ ]:
# ── Cell 10: Audience Segment Allocation ──────────────────────
# Second-level split: distribute a platform's budget across audience segments.
# Cross-platform allocator decides Meta's total; AudienceConstraintManager splits it.

segments = [
    AudienceSegment(
        segment_id='retargeting_warm', segment_name='Warm Retargeting',
        platform='meta',
        min_budget_pct=0.20, max_budget_pct=0.50,
        max_exposures_per_user=5, max_daily_frequency=2,
    ),
    AudienceSegment(
        segment_id='lookalike_1pct', segment_name='1% Lookalike',
        platform='meta',
        min_budget_pct=0.10, max_budget_pct=0.60,
        max_exposures_per_user=10, max_daily_frequency=3,
    ),
    AudienceSegment(
        segment_id='broad_prospecting', segment_name='Broad Prospecting',
        platform='meta',
        min_budget_pct=0.05, max_budget_pct=0.40,
        max_exposures_per_user=8, max_daily_frequency=3,
    ),
]

seg_manager = AudienceConstraintManager(segments)  # validates min/max budget pct sum <= 1

meta_results = result.platform_campaign_results.get('meta', [])
meta_action = meta_results[0].action if meta_results else ActionSpace()  # DRL audience action (EXPAND/REFINE modifies allocation)
meta_budget = 8000.0

seg_result = seg_manager.allocate_budget(  # score = 0.5*ROAS + 0.3*CVR + 0.2*CTR; higher score → more budget
    platform_budget=meta_budget,
    action=meta_action,
    performance_signals={
        'retargeting_warm': {'roas': 5.2, 'cvr': 0.08, 'ctr': 0.06},
        'lookalike_1pct':   {'roas': 3.1, 'cvr': 0.04, 'ctr': 0.035},
        'broad_prospecting':{'roas': 1.5, 'cvr': 0.02, 'ctr': 0.02},
    },
)

print('AUDIENCE SEGMENT ALLOCATION (Meta) ')
seg_rows = []
for alloc in seg_result.allocations:
    seg_rows.append({
        'Segment': alloc.segment_id,
        'Budget %': f'{alloc.recommended_budget_pct:.1%}',
        'Budget $': f'${meta_budget * alloc.recommended_budget_pct:,.0f}',
        'Freq Cap': alloc.recommended_daily_frequency,
        'Rationale': alloc.rationale,
    })
display(pd.DataFrame(seg_rows))

print(f'\nConstraints passed: {seg_result.total_budget_check_passed}')
if seg_result.violations:
    print(f'Violations: {seg_result.violations}')
print(f'\nNarrative: {seg_result.narrative}')

AUDIENCE SEGMENT ALLOCATION (Meta) 


,Segment,Budget %,Budget $,Freq Cap,Rationale
0,retargeting_warm,50.0%,"$4,000",2,"Allocated 50.0% ($4,000) to Warm Retargeting b..."
1,lookalike_1pct,25.5%,"$2,043",3,"Allocated 25.5% ($2,043) to 1% Lookalike based..."
2,broad_prospecting,9.9%,$796,3,Allocated 9.9% ($796) to Broad Prospecting bas...



Constraints passed: True

Narrative: Budget $8,000 distributed across 3 segment(s):
- Warm Retargeting: 50.0% of budget ($4,000), frequency cap 2/day
- 1% Lookalike: 25.5% of budget ($2,043), frequency cap 3/day
- Broad Prospecting: 9.9% of budget ($796), frequency cap 3/day


### Reading the segment allocation above

Second-level budget split:
1. Cross-platform allocator decides Meta's total budget.
2. `AudienceConstraintManager` distributes that budget across 3 audience segments.

Scoring: `score = 0.5*ROAS + 0.3*CVR + 0.2*CTR`.
- **Retargeting** (ROAS 5.2x) → highest score → most budget (up to 50% cap).
- **Broad Prospecting** (ROAS 1.5x) → lowest score → floor allocation (5% min).
- DRL audience action modifies allocation: EXPAND boosts top segment, EXCLUDE zeros bottom.

In [ ]:
# ── Cell 11: Verification Steps ────────────────────────────────────
# Use this cell to verify segment allocation when you change the platform.
#
# STEPS TO VERIFY:
# 1. In Cell 10, change platform='meta' to your target (e.g. 'google', 'tiktok', 'amazon', 'linkedin')
#    in all three AudienceSegment definitions.
# 2. Update meta_results → platform_results and meta_budget → use lookup below.
# 3. Re-run Cell 10, then run this verification cell.
# 4. Check: platform exists in results, budget matches allocation table, constraints passed.

PLATFORM = 'meta'  # Change to match what you set in Cell 10 (e.g. 'google', 'tiktok', 'amazon', 'linkedin')

# Step 1: Verify platform has optimization results
if PLATFORM in result.platform_campaign_results:
    n_campaigns = len(result.platform_campaign_results[PLATFORM])
    print(f"✓ Step 1: Found {n_campaigns} campaign(s) for '{PLATFORM}'")
else:
    print(f"✗ Step 1: No results for '{PLATFORM}'. Available: {list(result.platform_campaign_results.keys())}")

# Step 2: Get platform's recommended budget from allocation results
platform_alloc = next((a for a in result.allocations if a.platform == PLATFORM), None)
if platform_alloc:
    platform_budget = platform_alloc.recommended_budget
    print(f"✓ Step 2: {PLATFORM} recommended budget = ${platform_budget:,.0f}")
else:
    platform_budget = 0.0
    print(f"✗ Step 2: No allocation for '{PLATFORM}'")

# Step 3: Verify segment allocation used correct platform (compare to Cell 10 output)
# If you used platform_budget from above in Cell 10, the narrative should show this amount.
print(f"✓ Step 3: Cell 10 should show 'AUDIENCE SEGMENT ALLOCATION ({PLATFORM.upper()})' and budget ${platform_budget:,.0f}")

# Step 4: Summary
print(f"\n--- Verification Summary ---")
print(f"Platform: {PLATFORM}")
print(f"Budget:   ${platform_budget:,.0f}")
print(f"Check:   Constraints passed = True in Cell 10 output")

✓ Step 1: Found 1 campaign(s) for 'meta'
✓ Step 2: meta recommended budget = $3,943
✓ Step 3: Cell 10 should show 'AUDIENCE SEGMENT ALLOCATION (META)' and budget $3,943

--- Verification Summary ---
Platform: meta
Budget:   $3,943
Check:   Constraints passed = True in Cell 10 output


## XPCO Demo Summary

| Step | Component | Output |
|------|-----------|--------|
| 1 | Portfolio Snapshot | Aggregated spend/revenue per platform |
| 2 | Marginal ROAS Estimation | Per-platform marginal return (from 30 days of history) |
| 3 | Budget Allocation | Shift % and $ per platform (Amazon ↑, LinkedIn ↓) |
| 4 | Per-Campaign DRL | bid/budget adj, audience/creative action per campaign |
| 5 | Mock LLM Creative | headline, body, CTA, offer per campaign |
| 6 | xAI Narrative | situation, decision, reasoning per campaign |
| 7 | Segment Allocation | Budget % per audience segment within platform |

Every output layer is visible in this notebook.